In [ ]:
# ==============================================================
# STEP —  Create unified JSON   (Heart + Micro + Lab + Procedure)
# ==============================================================

import json
import pandas as pd
from pprint import pprint

# -------- Load preprocessed sources --------------
with open("../features/OUTPUT_JSON_HEART.json", "r", encoding="utf-8") as f:
    heart_data = json.load(f)

with open("../features/OUTPUT_JSON_MICROBIOLOGY", "r", encoding="utf-8") as f:
    micro_data = json.load(f)

# teammate’s features
lab_df = pd.read_csv("../features/laboratory_tests_features.csv")
proc_df = pd.read_csv("../features/procedure_code_features.csv")

In [2]:
# -------------------------------------------------
# Convert laboratory and procedure feature tables to JSON keyed by hadm_id
# -------------------------------------------------
lab_json = lab_df.set_index("hadm_id").apply(lambda r: r.dropna().to_dict(), axis=1).to_dict()
proc_json = proc_df.set_index("hadm_id").apply(lambda r: r.dropna().to_dict(), axis=1).to_dict()

print(f"Lab records: {len(lab_json)}  |  Procedure records: {len(proc_json)}")

Lab records: 4751  |  Procedure records: 3459


In [3]:
# -------------------------------------------------
# Combine under each subject / admission
# -------------------------------------------------
combined_all = {
    "created_at": heart_data.get("created_at"),
    "subjects": {}
}

subject_ids = set(heart_data.get("subjects", {}).keys()) | set(micro_data.get("subjects", {}).keys())

for sid in subject_ids:
    subj_struct = {"subject_id": sid}

    # ―― Heart dataset ――
    heart_part = heart_data.get("subjects", {}).get(sid, {}).get("heart_dataset", None)
    subj_struct["heart_dataset"] = heart_part

    # ―― Microbiology dataset ――
    micro_part = micro_data.get("subjects", {}).get(sid, {}).get("microbiology_dataset", None)
    subj_struct["microbiology_dataset"] = micro_part

    # ―― Admission mapping for new tables ――
    # build mapping of hadm ids already recorded in either dataset
    existing_hadms = set()
    if heart_part:
        existing_hadms |= set(heart_part.get("hadm_records", {}).keys())
    if micro_part:
        existing_hadms |= set(micro_part.get("hadm_records", {}).keys())

    full_hadm = {}
    for hadm in existing_hadms:
        hadm_key = str(hadm)
        full_hadm[hadm_key] = {
            "hadm_id": int(hadm_key),
            "lab_dataset": lab_json.get(int(hadm_key), None),
            "procedure_dataset": proc_json.get(int(hadm_key), None)
        }
    subj_struct["hadm_records_all"] = full_hadm if full_hadm else None

    combined_all["subjects"][sid] = subj_struct

In [4]:
# -------------------------------------------------
# Statistics — coverage of datasets
# -------------------------------------------------
coverage_stats = {"all4": 0, "heart_micro_only": 0, "heart_only": 0, "micro_only": 0, "partial": 0}
detail = []

for sid, sdata in combined_all["subjects"].items():
    h = bool(sdata["heart_dataset"])
    m = bool(sdata["microbiology_dataset"])

    # gather hadm-level presence for lab/proc
    hadm_struct = sdata.get("hadm_records_all") or {}
    lab_present = any(hadm_struct[h]["lab_dataset"] is not None for h in hadm_struct)
    proc_present = any(hadm_struct[h]["procedure_dataset"] is not None for h in hadm_struct)

    count_present = sum([h, m, lab_present, proc_present])

    if count_present == 4:
        coverage_stats["all4"] += 1
    elif count_present == 2 and h and m:
        coverage_stats["heart_micro_only"] += 1
    elif count_present == 1 and h:
        coverage_stats["heart_only"] += 1
    elif count_present == 1 and m:
        coverage_stats["micro_only"] += 1
    else:
        coverage_stats["partial"] += 1

    detail.append({
        "subject_id": sid,
        "heart": h,
        "micro": m,
        "lab": lab_present,
        "procedure": proc_present,
        "datasets_available": count_present
    })

cov_df = pd.DataFrame(detail)
print("\nDataset presence summary:")
print(pd.DataFrame.from_dict(coverage_stats, orient="index", columns=["count"]))
print("\nExample of subject-level availability:")
print(cov_df.head())


Dataset presence summary:
                  count
all4               1850
heart_micro_only      0
heart_only            7
micro_only            0
partial            2719

Example of subject-level availability:
  subject_id  heart  micro   lab  procedure  datasets_available
0   19997487  False   True  True       True                   3
1   12559662   True  False  True      False                   2
2   17086599   True  False  True       True                   3
3   13757356   True  False  True       True                   3
4   19919213   True  False  True      False                   2


In [6]:
# -------------------------------------------------
# Save final unified JSON
# -------------------------------------------------
with open("OUTPUT_JSON_ALL_FOUR.json", "w", encoding="utf-8") as f:
    json.dump(combined_all, f, indent=2, ensure_ascii=False)
print("\nCombined 4-dataset JSON saved to OUTPUT_JSON_ALL_FOUR.json")


Combined 4-dataset JSON saved to OUTPUT_JSON_ALL_FOUR.json


## Checking out the structure of one subject_id with all four Datasets present: 15198228

In [9]:
cov_df

,subject_id,heart,micro,lab,procedure,datasets_available
0,19997487,False,True,True,True,3
1,12559662,True,False,True,False,2
2,17086599,True,False,True,True,3
3,13757356,True,False,True,True,3
4,19919213,True,False,True,False,2
...,...,...,...,...,...,...
4571,19997644,False,True,True,True,3
4572,10931276,True,True,True,False,3
4573,15198228,True,True,True,True,4
4574,11532808,True,False,True,True,3


In [12]:
import json
from pprint import pprint

with open("OUTPUT_JSON_ALL_FOUR.json", "r") as f:
    data = json.load(f)

subject_15198228 = data["subjects"].get('15198228', None)
if subject_15198228:
    pprint(subject_15198228)
else:
    print("Subject with ID '15198228' not found.")

{'hadm_records_all': {'28287873': {'hadm_id': 28287873,
                                   'lab_dataset': {'cl_abnormal_ratio': 0.4285714285714285,
                                                   'cl_max': 105.0,
                                                   'cl_min': 92.0,
                                                   'cl_n': 14.0,
                                                   'cl_n_abnormal': 6.0,
                                                   'creat_abnormal_ratio': 1.0,
                                                   'creat_max': 4.4,
                                                   'creat_min': 3.3,
                                                   'creat_n': 14.0,
                                                   'creat_n_abnormal': 14.0,
                                                   'glu_abnormal_ratio': 1.0,
                                                   'glu_max': 387.0,
                                                   'glu_min': 112.0,
